# **1. Object Detection**

Object detection(객체 탐지)은 이미지나 영상에서 **특정 객체의 존재 여부를 확인하고, 해당 객체의 위치를 바운딩 박스(Bounding Box)로 표시하는 기술**이다. 이는 컴퓨터 비전에서 중요한 분야로, 이미지 내에서 여러 개의 객체를 동시에 탐지하고 분류할 수 있다. Object detection은 주로 딥러닝 기반의 CNN(합성곱 신경망) 모델을 활용하며, 대표적인 알고리즘으로는 R-CNN 계열(Faster R-CNN, Mask R-CNN), YOLO(You Only Look Once), SSD(Single Shot MultiBox Detector) 등이 있다. 이러한 기술은 자율 주행, 보안 감시, 의료 영상 분석, 증강 현실 등 다양한 분야에서 활용된다.

### Object Detection 논문 추천 리스트
![논문 추천 리스트](./images/논문%20추천%20리스트.png)

### **1. 객체 탐지의 출력**
- 클래스: 사람, 자동차, 개 등
- 바운딩 박스: 객체의 위치와 크기
- confidence score: 해당 예측을 모델이 얼마나 확신하는지 나타내는 점수
> 객체 탐지는 이미지 전체에 하나의 클래스만 붙이는 Image Classification과 다르다.
> **한 이미지 안에 여러 객체가 있어도 각각 따로 찾아낼 수 있다.**

### **2. 대표적인 딥러닝 기반 계열**
1. Two-Stage: Faster R-CNN, Mask R-CNN 등
    - 객체 후보 영역을 만들고 후보를 다시 분류하고 박스를 보정한다.
    - 일반적으로 정밀한 탐지에 강점이 있다.
2. One-Stage: YOLO, SSD, RetinaNet 등
    - 한번의 네트워크 흐름에서 객체 클래스와 위치를 직접 예측한다.
    - 일반적으로 빠르며, 실시간 응용에 많이 사용한다.
> 실무에서는 자율주행, CCTV 분석, 제조 분량 검출, 의료영상 분석 등에서 사용한다.
> 다만 현대 모델에서는 "One-Stage는 항상 부정확하고 Two-Stage는 항상 정확하다."라고 단정하면 안된다.
> 데이터셋, 모델 크기, 입력 해상도, 학습 방식에 따라 성능관계는 달라진다.

# **2. YOLO**

YOLO(You Only Look Once)는 이미지를 **격자 단위로 나누어 각 격자에서 객체의 위치(바운딩 박스 좌표)와 클래스를 동시에 예측하는 1단계 객체 탐지(One-Stage Detection) 알고리즘**이다. 
(최신 YOLO 계열을 설명할 때는 단순히 격자로 나눔이 아닌 다중 스케일(feature map)의 여러 위치에서 좌표를 통해 11객체 후보를 동시에 예측하는 방법을 사용)

**한 번의 신경망 추론으로 전체 이미지의 탐지를 수행하기 때문에 매우 빠르고, 실시간 객체 탐지에 적합하다는 장점**이 있다. YOLO 계열은 v1에서 시작해 v3까지는 Joseph Redmon이 개발했으며, 이후 v4·v7은 커뮤니티 연구자들, v5·v8·v11은 Ultralytics가 주도적으로 발전시켜 현재까지 이어지고 있다. 이러한 발전 과정을 거치면서 YOLO는 속도와 정확도의 균형을 잡은 대표적인 객체 탐지 모델로 자리 잡아 다양한 산업 현장에서 활용되고 있다.

### **1. Ultralytics 모델의 크기 표기**
- n(nano): 가장 가볍고 빠름. 실시간 실험에 유리
- s(small): 속도와 정확도의 균형이 좋아 실무 프로토타입에 적합
- m(medium): 더 높은 정확도를 기대할 수 있지만 연산량이 증가함
- l(large): 정확도 중심
- x(ectra-large): 가장 무겁고 높은 연산
> 모델이 커지면 무조건 좋은 것은 아니다.
> 실제 프로젝트에서는 GPU, FPS, 데이터 크기, 목표 정확도를 함꼐 고려해야 한다.

### **2. Ultralytics 주요 task**
- Detect: 객체 위치를 axis-aligned bounding box로 탐지
- Segment: 객체 별 mask를 예측하는 instance segmentation
- Pose: keypoint를 예측하는 pose estimation. 기본 pretrained 모델은 사람 관절 추정에 많이 사용되지만 custom keypoint 데이터도 학습 가능
- Classify: 이미지 전체를 하나의 클래스 또는 설정된 분류 문제로 예측
- OBB(Oriented Bounding Box): 객체의 방향까지 포함한 회전된 바운딩 박스를 예측(박스 자체에 angle이 포함)

In [ ]:
import sys
import torch
from pathlib import Path

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd()
print('프로젝트 경로:', PROJECT_ROOT)

# **3. PascalVOC2007**

PascalVOC 2007은 객체 탐지(Object Detection), 분할(Segmentation), 동작 인식(Action Recognition) 등의 다양한 컴퓨터 비전 과제를 위한 **벤치마크 데이터셋**이다. 총 20개의 객체 클래스(예: 사람, 자동차, 개, 고양이 등)를 포함하며, 훈련(train), 검증(val), 테스트(test) 세트로 구성되어 있다. 각 이미지에는 객체의 경계 상자(Bounding Box) 및 해당 클래스 레이블이 주어지며, Mean Average Precision(mAP) 평가 기준을 적용하여 성능을 측정한다.

In [ ]:
VOC_ROOT = PROJECT_ROOT / 'data' / 'VOCdevkit' / 'VOC2007'

ANNOTATIONS_DIR = VOC_ROOT / 'Annotations'
IMAGES_DIR = VOC_ROOT / 'JPEGImages'
IMAGESETS_DIR = VOC_ROOT / 'ImageSets' / 'Main'

if VOC_ROOT.exists():
    print('VOC_ROOT:', VOC_ROOT)

In [ ]:
# VOC 데이터를 변환한 결과를 저장할 기준 폴더
VOC2YOLO_ROOT = Path('./data/VOC2YOLO')

# YOLO 학습용으로 변환된 VOC 데이터가 들어갈 최종 폴더
YOLO_ROOT = VOC2YOLO_ROOT / 'YOLO'

# Ultralytics가 일반적으로 사용하는 images/labels 구조를 미리 생성
# train / val / test 각각 이미지 폴더와 라벨 폴더를 분리
for sub in [
    'images/train2007', 'images/val2007', 'images/test2007',
    'labels/train2007', 'labels/val2007', 'labels/test2007'
]:
    (YOLO_ROOT / sub).mkdir(parents=True, exist_ok=True)
    # 실행 위치를 기준으로 절대 경로로 바꿔줌
print('dataset root/:', VOC2YOLO_ROOT.resolve())

# **4. YOLO annotation 기본 구조**

```
예시) 6 0.189180 0.812613 0.378360 0.374775
<class_id> <x_center> <y_center> <width> <height>
```
- class_id: 0부터 시작하는 클래스 번호
- x_center, y_center: 박스 중심점
- width, height: 박스의 너비와 높이
- 좌표는 모두 이미지 크기로 나눈 0 ~ 1로 정규화된 값
> VOC XML의 (xmin, ymin, xmax, ymax)를 YOLO 형식으로 바꿀 때는 중심점과 크기를 계산한 뒤 이미지 너비와 높이로 정규화해야 한다.

In [ ]:
import xml.etree.ElementTree as ET
import shutil

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID

In [ ]:
def voc_to_yolo(x_min, y_min, x_max, y_max, image_w, image_h):
    x_center = ((x_min + x_max)/2.0) / image_w
    y_center = ((y_min + y_max)/2.0) / image_h
    box_w = (x_max - x_min)/image_w
    box_h = (y_max - y_min)/image_h
    return x_center, y_center, box_w, box_h

In [ ]:
def xml_to_txt(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')  # find: width/height를 담은 부모 요소라서 Element 객체로 받음
    image_w = float(size.findtext('width'))  
    image_h = float(size.findtext('height'))

    lines = []
    for obj in root.findall('object'):  # findall: 조건에 맞는 태그를 전부 리스트로 반환(object가 여러 개일 수 있음)
        class_name = obj.findtext('name') # findtext: leaf 요소의 텍스트 값을 바로 반환
        if class_name not in CLASS_TO_ID:
            continue

        difficult = int(obj.findtext('difficult', default=0))
        if difficult == 1: # 객체를 사람이 봐도 판별하기 어려운 경우를 나타냄
            continue

        bnd = obj.find('bndbox')  # find: 자식 요소(xmin/ymin/xmax/ymax)를 가진 부모 요소(bndbox)를 Element 객체를 추출함
        x_min = float(bnd.findtext('xmin'))
        y_min = float(bnd.findtext('ymin'))
        x_max = float(bnd.findtext('xmax'))
        y_max = float(bnd.findtext('ymax'))

        x_min = max(0.0, min(x_min, image_w))
        y_min = max(0.0, min(y_min, image_h))
        x_max = max(0.0, min(x_max, image_w))
        y_max = max(0.0, min(y_max, image_h))

        if x_max <= x_min or y_max <= y_min:
            continue

        x, y, w, h = voc_to_yolo(x_min, y_min, x_max, y_max, image_w, image_h)

        class_id = CLASS_TO_ID[class_name]

        lines.append(f'{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text('\n'.join(lines), encoding='utf-8')

In [ ]:
def prepare_split(split_name, output_name):
    split_file =  IMAGESETS_DIR/ f'{split_name}.txt'
    image_dir =  IMAGES_DIR
    annotation_dir = ANNOTATIONS_DIR

    yolo_image_dir = YOLO_ROOT / 'images' / output_name
    yolo_label_dir = YOLO_ROOT / 'labels' / output_name
    yolo_image_dir.mkdir(parents=True, exist_ok=True)
    yolo_label_dir.mkdir(parents=True, exist_ok=True)

    image_ids = [x.strip() for x in split_file.read_text(encoding='utf-8').splitlines() if x.strip()]

    for image_id in image_ids:
        src_image = image_dir / f'{image_id}.jpg'
        src_xml = annotation_dir / f'{image_id}.xml'

        shutil.copy2(src_image, yolo_image_dir / src_image.name)
        xml_to_txt(src_xml, yolo_label_dir / f'{image_id}.txt')
    print(f'{output_name}: {len(image_ids):,} images prepared')

In [ ]:
if YOLO_ROOT.exists():
    prepare_split('train', 'train2007')
    prepare_split('val', 'val2007')
else:
    print('YOLO_ROOT가 없습니다. 먼저 VOC2007 trainval 데이터를 준비하세요.')

In [ ]:
if YOLO_ROOT.exists():
    prepare_split('test', 'test2007')
else:
    print('YOLO_ROOT가 없습니다. 먼저 VOC2007 test 데이터를 준비하세요.')

# **5. Dataset YAML**

Ultralytics의 dataset YAML은 **데이터셋의 위치와 클래스 이름을 모델에게 알려주는 설정 파일**이다.
- Dataset YAML: train/val/test 경로, 클래스 이름
- Model YAML: 모델 architecture 정의
> 학습할 때 항상 model YAML을 직접 만들어야 하느 것은 아니다.
> 일반적인 transfer learning에서는 pretrained.pth 가중치를 불러오고 dataset YAML만 지정하면 편하게 사용할 수 있다.
```
pip install ultralytics
pip install pyyaml
```

In [ ]:
import yaml
import torch
from ultralytics import YOLO

In [ ]:
custom_voc_yaml = YOLO_ROOT / 'custom_voc.yaml'

data_config = {
    'path': str(YOLO_ROOT.resolve()),
    'train': 'images/train2007',
    'val': 'images/val2007',
    'test': 'images/test2007',
    'names': {i: name for i, name in enumerate(VOC_CLASSES)}
}

with open(custom_voc_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False)

print('생성된 YAML 경로:', custom_voc_yaml)
print(custom_voc_yaml.read_text(encoding='utf-8'))

# **6. Pretrained 모델 선택**

In [ ]:
# .pt: 모델 구조+가중치를 통째로 저장
# .pth: 가중치(state_dict)만 저장
model = YOLO('yolo11s.pt')
model

In [ ]:
# model train: 학습을 시작하는 메서드
results = model.train(
    data=str(custom_voc_yaml),
    epochs=5,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    project=str(PROJECT_ROOT / "runs" / "detect"),
    name="voc2007_yolo11s",
    exist_ok=True,
    seed=2026
)

# **7. 객체 탐지 성능 평가 지표**

- 객체 탐지는 단순한 accuracy 하나만으로 성능을 평가하기 어렵다.
- 분류 문제에서는 보통 클래스가 맞았는 지 확인하면 되지만, 객체탐지에서는 아래 두가지를 동시에 확인해야 한다.
    1. 무엇을 찾았는 가?: 예측 클래스가 정답 클래스와 같은지 비교
    2. 어디를 찾았는 가?: 예측한 bounding box가 실제 객체 위치와 충분히 겹치는 지

### 객체 탐지 평가
```
예측 결과 생성 > Confidence 기준 적용 > 정답 박스와 예측 박스 매칭 > IoU 확인 > TP/FP/FN 결정 > Precision/Recall 계산 > PR curve > AP > 클래스 평균(mAP)
```
> 위 흐름을 이해하면 Precision, Recall, IoU, AP, mAP가 서로 따로 떨어진 지표가 아니라 하나의 평가 과정 안에서 연겨뢴 값이라는 것을 알 수 있다.

### IoU(Intersection over Union)
IoU는 객체 탐지에서 **예측 bounding box와 정답 bounding box의 위치가 얼마나 비슷한 지 측정**하는 지표이다.
- Intersection(교집합): 예측 박스와 정답 박스가 실제로 겹치는 영역
- Union(합집합): 두 박스를 합쳤을 때 차지하는 전체 영역
- IoU=1: 예측 박스와 정답 박스가 완전히 일치
- IoU가 1에 가까움: 위치를 매우 정확하게 예측
- IoU가 0에 가까움: 두 박스가 조금만 겹치거나 위치가 매우 어긋남
> 객체 탐지에서는 클래스가 맞아도 bounding box가 객체와 거의 겹치지 않는다면 올바른 탐지라고 보기 어렵다.
> 따라서, 평가할 때 일정 IoU 이상을 만족한 예측만 TP로 인정한다.
> AP50의 50은 대표적으로 "IoU=0.50"을 기준으로 TP/FP를 판단하여 AP를 계산한다는 의미이다.

### IoU 계산 순서
Bounding Box가 (xmin, ymin, xmax, ymax) 형태라고 할 때 개념적으로 아래 순서와 같다.
1. 두 박스가 겹치는 영역의 좌표를 구한다.
2. 겹치는 영역의 너비와 높이를 계산한다.
3. 각각의 bounding box 면적을 계산한다.
    - 면적 = 겹치는 너비 * 겹치는 높이
4. 합집합 면적 = 박스1 면적 + 박스2 면적 - 교집합 면적
5. IoU = 교집합 면접 / 합집합 면적
> IoU는 박스의 크기 자체를 평가하는 것이 아니라, 정답 박스와 예측 박스의 상대적인 겹침 정도를 측정한다.

### AP(Average Precision)
AP는 한 클래스에 대해 모델의 **Precision과 Recall 성능을 하나의 값으로 요약**한 지표이다. 여러 Precision 값의 단순 평균이라고 생각하기 쉽지만, 객체 탐지에서는 **각 예측마다 confidence score**가 있디. 평가기는 높은 confidence 예측부터 낮은 confidence 예측까지 순서대로 보며 Precision과 Recall의 변화를 계산하고 PR Curve를 만들 수 있다.
    - 가로축: Recall
    - 세로축: Precision

### AP 계산 흐름
한 클래스만 놓고 보면 아래 순서로 평가가 진행한다.
1. 해당 클래스의 모든 예측 결과를 모은다.
2. Confidence가 높은 순서로 정렬한다.
3. 각 예측을 정답 박스와 비교한다.
4. 클래스와 IoU 조건을 이용해 TP 또는 FP를 결정한다.
5. 예측을 하나씩 포함시키면서 Precision과 Recall을 계산한다.
6. Pricision-Recall Curve를 구성한다.
7. PR Curve 전체를 요약하여 AP를 계산한다.

- AP50: IoU 기준을 0.50으로 두었을 때 계산한 AP
- AP75: IoU 기준을 0.75으로 두었을 때 계산한 AP
> IoU 기준이 높아질수록 bounding box의 위치가 더 정확해야 TP로 인정된다. 
> 따라서, 일반적으로 AP75가 AP50보다 더 엄격한 평가하는 지표이다.

### mAP(mean Average Precision)
AP는 보통 한 클래스의 성능을 나타낸다. 객체 탐지 데이터셋에는 여러 클래스가 있으므로, **클래스 별 AP를 구한 뒤 평균**을 내어 전체 모델 성능을 표현한 값이다.
- mAP50: IoU=0.50 기준에서 각 클래스의 AP를 구한 뒤 클래스 전체 평균을 낸 값(Bounding Box가 정답과 어느 정도만 겹쳐도 TP로 인정되기 때문에 상대적으로 점수가 높게 나올 수 있음)
- mAP50-95: IoU 기준을 0.50, 0.55, 0.60, ... 0.90, 0.95 각 IoU 기준에서 AP를 계산하고, 클래스와 IoU 기준 전체에 대해 평균을 냄
> mAP50-95는 단순히 객체를 대략 찾는 것뿐 아니라 bounding box의 위치를 얼마나 정밀하게 맞추는지까지 더 엄격하게 반영한다.

### 문제
mAP50은 높은데, mAP50-95가 상대적으로 많이 낮다면?
- 객체 자체는 잘 찾지만 bounding box 위치 정밀도가 부족할 가능성이 높다.

둘 다 낮다면?
- 분류, 위치 예측, 데이터 품질, 학습량 등 전체적인 점검 필요하다.
> 전체 mAP만 보지 말고 클래스별 AP도 꼭 함께 확인해야 한다. 
> 특정 클래스만 유독 성능이 낮을 수 있다.

### Confidence threshold와 IoU threshold를 구분
1. Confidence threshold
모델이 출력한 예측 중 **확신이 얼머나 높은 예측만 날길 것인 지 결정하는 값**이다.
- 높게 설정: 확실한 예측 중심
- 낮게 설정: 더 많은 후보 유지
> Precision과 Recall의 균형에 영향을 준다.
2. IoU threshold
**예측 박스와 정답 박스를 비교**하여 이 예측을 **TP로 인정할 만큼 위치가 정확한 가를 판단하는 기준값이**다.

### **NMS**의 IoU threshold
YOLO의 predict(...., iou=...) 또는 일부 검증 설정에서 등장하는 iou는 겹쳐 있는 예측 박스를 정리하는 NMS용 threshold이다.
- 평가 IoU: 예측 박스와 정답 박스를 비교
- NMS IoU: 여러 예측 박스끼리 비교하여 중복 탐지를 제거

### FPS와 Latency
- FPS(Frames Per Second): 1초에 처리 가능한 프레임 수
- Latency: 한 입력이 결과로 나오기까지 걸리는 시간
> 둘은 관련이 있지만 같은 개념은 아니다.
> 특히, batch inference, preprocessing/postprocessing 포함 여부, 비동기 pipeline 여부에 따라 FPS, latency 계산이 정확히 성립하지 않을 수 있다.

### 실무 성능 측정 조건
실무 성능 측정 시에는 아래 조건을 함께 기록해야 한다.
- GPU/CPU/NPU 종류
- batch size
- FP32/FP16/INT8 정밀도
- preprocessing/NMS 포함 여부
> 객체 탐지 모델은 정확도만 높다고 좋은 모델이 아닐 수 있다.
> 실제 서비스에서는 정확도 + 속도 + 메모리 사용량 + 배포 장비를 함께 고려해야 한다.

# **8. 검증(validation)에서 metrics를 읽는 법**
Ultralytics에서는 학습 후 model.val()를 사용해 validation 또는 test split의 성능을 평가할 수 있다.
    - validation set 평가: model.val(split='val')
    - test set 평가: dataset YAML에 "test:"가 정의 되어 있을 때, model.val(split='test')

검증이 끝나면 아래와 같은 객체 탐지 metrics를 확인한다.
    - Precision: 탐지했다고 한 것 중 올바른 비율
    - Recall: 실제 객체 중 찾아낸 비율
    - mAP50: IoU=0.50 기준의 클래스 평균 AP
    - mAP50-95: IoU=0.50 ~ 0.95 등 여러 기준을 평균한 더 엄격한 AP
> model.val(iou=0.5)에서 사용하는 iou인자를 "mAP를 IoU=0.5에서만 계산한다."라는 의미로 사용하면 안된다.
> 해당 매개변수는 예측 후처리/NMS 동작과 관련된 IoU threshold 설정으로 사용하는 것이다.

In [ ]:
# 학습 중 validation 성능이 가장 좋았던 가중치(best.pt)를 불러옴
best_path = Path('runs/detect/voc2007_yolo11s/weights/best.pt')
model = YOLO(str(best_path))

In [ ]:
val_results = model.val(
    data=str(custom_voc_yaml),
    split='val',
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2
)

In [ ]:
print(val_results)

In [ ]:
# Ultralytics detection 결과에서 자주 사용하는 mAP 지표
# 버전에 따라 객체 속성 이름이 달라질 수 있음
try:
    print('mAP50:', val_results.box.map50)
    print('mAP50-95:', val_results.box.map)
    print('mAP75:', val_results.box.map75)
except AttributeError:
    print('설치된 Ultralytics 버전의 metrics 객체 구조를 확안하세요.')

In [ ]:
test_results = model.val(
    data=str(custom_voc_yaml),
    split='test',
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2
)
print(test_results)

# **9. 객체 탐지 추론**

predict()는 학습된 모델로 새로운 이미지/폴더/영상 등에 객체 탐지를 수행할 때 사용한다.
- conf: 너무 낮은 confidence 예측을 제거하는 임계값
- iou: NMS 단계에서 겹치는 예측을 정리할 때 사용하는 IoU threshold
- save=True: 시각화 된 결과 저장
- save_conf=True: txt 결과에 confidenc도 함께 저장
> conf를 올리면 일반적으로 더 확실한 예측만 남아 Precision은 좋아질 수 있지만 Recall은 낮아질 수 있다.

In [ ]:
source_dir = YOLO_ROOT / 'images' / 'test2007'

pred_results = model.predict(
    source=str(source_dir),
    imgsz=640,
    # confidence가 0.25보다 낮은 예측은 최종 결과에서 제거함 << 이 값을 높이면 더 확실한 예측만 남고, 낮추면 더 많은 후보가 남음
    conf=0.25,
    device=DEVICE,
    # Bounding Box와 클래스 이름이 그려진 결과 이미지를 저장
    save=True,
    # 예측 Bouding Box 정보를 YOLO txt 형식으로 저장
    save_txt=True,
    # txt 결과에 confidence score도 함께 기록
    save_conf=True,
    project=str(PROJECT_ROOT / 'runs' / 'detect'),
    name='voc2007_predict',
    exist_ok=True
)

# pred_result에는 이미지별 탐색 결과 객체가 들어있음
print(f'예측 이미지 수: {len(pred_results)}')

In [ ]:
source_dir = YOLO_ROOT / 'images' / 'custom_images'

pred_results2 = model.predict(
    source=str(source_dir),
    imgsz=640,
    # confidence가 0.25보다 낮은 예측은 최종 결과에서 제거함 << 이 값을 높이면 더 확실한 예측만 남고, 낮추면 더 많은 후보가 남음
    conf=0.25,
    device=DEVICE,
    # Bounding Box와 클래스 이름이 그려진 결과 이미지를 저장
    save=True,
    # 예측 Bouding Box 정보를 YOLO txt 형식으로 저장
    save_txt=True,
    # txt 결과에 confidence score도 함께 기록
    save_conf=True,
    project=str(PROJECT_ROOT / 'runs' / 'detect'),
    name='voc2007_predict2',
    exist_ok=True
)

# pred_result에는 이미지별 탐색 결과 객체가 들어있음
print(f'예측 이미지 수: {len(pred_results2)}')